In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [3]:
dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "val.jsonl"
    }
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [4]:
def format_example(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    }

dataset = dataset.map(format_example)

Map:   0%|          | 0/696 [00:00<?, ? examples/s]

Map:   0%|          | 0/175 [00:00<?, ? examples/s]

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [6]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",

    fp16=True,
    bf16=False,

    gradient_checkpointing=True
)

In [11]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args
)

In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",

    fp16=True,
    bf16=False,

    gradient_checkpointing=True
)

In [14]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args
)

In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",

    fp16=False,
    bf16=False,

    gradient_checkpointing=True
)

In [17]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args
)

In [18]:
trainer.train()

Step,Training Loss
10,2.359457
20,1.252009
30,0.581829
40,0.341246
50,0.273190
60,0.189643
70,0.203407
80,0.189920
90,0.180218
100,0.193292


TrainOutput(global_step=522, training_loss=0.25159005164186615, metrics={'train_runtime': 181.3125, 'train_samples_per_second': 11.516, 'train_steps_per_second': 2.879, 'total_flos': 812931727835136.0, 'train_loss': 0.25159005164186615})

In [19]:
model.save_pretrained("adapters")
tokenizer.save_pretrained("adapters")

('adapters/tokenizer_config.json',
 'adapters/chat_template.jinja',
 'adapters/tokenizer.json')

In [20]:
model.save_pretrained("adapters")
tokenizer.save_pretrained("adapters")

('adapters/tokenizer_config.json',
 'adapters/chat_template.jinja',
 'adapters/tokenizer.json')

In [22]:
!zip -r adapters.zip adapters

  adding: adapters/ (stored 0%)
  adding: adapters/adapter_config.json (deflated 57%)
  adding: adapters/adapter_model.safetensors (deflated 22%)
  adding: adapters/tokenizer.json (deflated 85%)
  adding: adapters/README.md (deflated 65%)
  adding: adapters/chat_template.jinja (deflated 60%)
  adding: adapters/tokenizer_config.json (deflated 46%)


In [23]:
from google.colab import files
files.download("adapters.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# base model load
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(base_model)

# adapter load
model = PeftModel.from_pretrained(model, "adapters")

model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_fe

In [26]:
prompt = """### Instruction:
Answer the finance question

### Input:
What is GST?

### Response:
"""

In [27]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7
)

print(tokenizer.decode(output[0]))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<s> ### Instruction:
Answer the finance question

### Input:
What is GST?

### Response:
GST is an indirect tax applied on goods and services.</s>


In [28]:
prompt = """### Instruction:
Solve the problem step by step

### Input:
If GST is 18% on ₹500, what is total?

### Response:
"""

In [30]:
import torch

def run_test(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7
    )

    print("\n-------------------------------")
    print("PROMPT:")
    print(prompt)
    print("\nOUTPUT:")
    print(tokenizer.decode(output[0]))
    print("-------------------------------\n")


# QA TEST
prompt1 = """### Instruction:
Answer the finance question

### Input:
What is GST?

### Response:
"""

# REASONING TEST
prompt2 = """### Instruction:
Solve the problem step by step

### Input:
If GST is 18% on ₹500, what is total?

### Response:
"""

# EXTRACTION TEST
prompt3 = """### Instruction:
Extract the tax amount from the sentence

### Input:
The GST applied is ₹250

### Response:
"""

# RUN ALL
run_test(prompt1)
run_test(prompt2)
run_test(prompt3)


-------------------------------
PROMPT:
### Instruction:
Answer the finance question

### Input:
What is GST?

### Response:


OUTPUT:
<s> ### Instruction:
Answer the finance question

### Input:
What is GST?

### Response:
GST is an indirect tax applied on goods and services.</s>
-------------------------------


-------------------------------
PROMPT:
### Instruction:
Solve the problem step by step

### Input:
If GST is 18% on ₹500, what is total?

### Response:


OUTPUT:
<s> ### Instruction:
Solve the problem step by step

### Input:
If GST is 18% on ₹500, what is total?

### Response:
GST = 90.0, Total = 600</s>
-------------------------------


-------------------------------
PROMPT:
### Instruction:
Extract the tax amount from the sentence

### Input:
The GST applied is ₹250

### Response:


OUTPUT:
<s> ### Instruction:
Extract the tax amount from the sentence

### Input:
The GST applied is ₹250

### Response:
250</s>
-------------------------------

